In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path("AReM")
activities = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]
for activity in activities:
    files = list((data_path / activity).glob("*.csv"))
    print(f"Activity: {activity}, Number of files: {len(files)}")

Activity: bending1, Number of files: 7
Activity: bending2, Number of files: 6
Activity: cycling, Number of files: 15
Activity: lying, Number of files: 15
Activity: sitting, Number of files: 15
Activity: standing, Number of files: 15
Activity: walking, Number of files: 15


In [2]:
#1b 
train_files = []
test_files = []
for activity in activities:
    folder = data_path / activity
    files = sorted(folder.glob("*.csv"),
                   key=lambda f: int(f.stem.replace("dataset", "")))

    if activity in ["bending1", "bending2"]:
        test_files.extend(files[:2])
        train_files.extend(files[2:])
    else:
        test_files.extend(files[:3])
        train_files.extend(files[3:])
print("Training files:", len(train_files))
print("Testing files:", len(test_files))

# figuring out shape
sample_file = train_files[0]
df = pd.read_csv(sample_file, comment="#",header=None)
print("File", sample_file)
print("Shape", df.shape)
display(df.head())
    

Training files: 69
Testing files: 19
File AReM/bending1/dataset3.csv
Shape (480, 7)


,0,1,2,3,4,5,6
0,0,42.00,0.71,21.25,0.43,30.00,0.00
1,250,41.50,0.50,20.25,1.48,31.25,1.09
2,500,41.50,0.50,14.25,1.92,33.00,0.00
3,750,40.75,0.83,15.75,0.43,33.00,0.00
4,1000,40.00,0.71,20.00,2.74,32.75,0.43


In [3]:
def load_arem_file(file):
    # Read the file and remove trailing commas
    with open(file, "r") as f:
        lines = [
            line.rstrip("\r\n").rstrip(",")
            for line in f
            if line.strip() and not line.lstrip().startswith("#")
        ]

    # Handle comma-separated and whitespace-separated files
    from io import StringIO

    text = "\n".join(lines)

    if "," in lines[0]:
        df = pd.read_csv(StringIO(text), header=None)
    else:
        df = pd.read_csv(StringIO(text), header=None, sep=r"\s+")

    return df

In [4]:
#1c. ii
sensor_data = df.iloc[:, 1:]
print("Sensor data shape:", sensor_data.shape)
features = sensor_data.agg(['mean', 'std', 'min', 'max', 'median'])
print(features)

#q1, q3
q1 = sensor_data.quantile(0.25)
q3 = sensor_data.quantile(0.75)
print("Q1:", q1)    
print("Q3:", q3)

features.loc["q1"] = q1
features.loc["q3"] = q3
display(features)

feature_rows= []
for file in train_files + test_files:
    df = load_arem_file(file)
    sensor_data = df.iloc[:, 1:]
    row = {}
    for i, col in enumerate(sensor_data.columns, start=1):
        values = sensor_data[col]
        row[f"sensor_{i}_mean"] = values.mean()
        row[f"sensor_{i}_std"] = values.std()
        row[f"sensor_{i}_min"] = values.min()
        row[f"sensor_{i}_max"] = values.max()
        row[f"sensor_{i}_median"] = values.median()
        row[f"sensor_{i}_q1"] = values.quantile(0.25)
        row[f"sensor_{i}_q3"] = values.quantile(0.75)
    feature_rows.append(row)
feature_df = pd.DataFrame(feature_rows)
print("Feature DataFrame shape:", feature_df.shape)
display(feature_df.head())
print(feature_df.shape) # confirming shape is good to go

Sensor data shape: (480, 6)
                1        2          3         4          5         6
mean    43.954500  0.42625  22.122354  0.497312  35.588458  0.493292
std      1.558835  0.33869   3.030943  0.550657   1.999604  0.513506
min     35.000000  0.00000   6.500000  0.000000  29.000000  0.000000
max     47.400000  1.70000  29.750000  4.440000  38.500000  1.790000
median  44.330000  0.47000  23.000000  0.430000  36.000000  0.430000
Q1: 1    43.0000
2     0.0000
3    19.7500
4     0.0000
5    35.3625
6     0.0000
Name: 0.25, dtype: float64
Q3: 1    45.00
2     0.50
3    24.00
4     0.83
5    36.50
6     0.94
Name: 0.75, dtype: float64


,1,2,3,4,5,6
mean,43.954500,0.42625,22.122354,0.497312,35.588458,0.493292
std,1.558835,0.33869,3.030943,0.550657,1.999604,0.513506
min,35.000000,0.00000,6.500000,0.000000,29.000000,0.000000
max,47.400000,1.70000,29.750000,4.440000,38.500000,1.790000
median,44.330000,0.47000,23.000000,0.430000,36.000000,0.430000
q1,43.000000,0.00000,19.750000,0.000000,35.362500,0.000000
q3,45.000000,0.50000,24.000000,0.830000,36.500000,0.940000


Feature DataFrame shape: (88, 42)


,sensor_1_mean,sensor_1_std,sensor_1_min,sensor_1_max,sensor_1_median,sensor_1_q1,sensor_1_q3,sensor_2_mean,sensor_2_std,sensor_2_min,...,sensor_5_median,sensor_5_q1,sensor_5_q3,sensor_6_mean,sensor_6_std,sensor_6_min,sensor_6_max,sensor_6_median,sensor_6_q1,sensor_6_q3
0,43.954500,1.558835,35.00,47.40,44.33,43.00,45.00,0.426250,0.338690,0.0,...,36.00,35.3625,36.50,0.493292,0.513506,0.0,1.79,0.43,0.00,0.94
1,42.179812,3.670666,33.00,47.75,43.50,39.15,45.00,0.696042,0.630860,0.0,...,35.00,30.4575,36.33,0.613521,0.524317,0.0,2.18,0.50,0.00,1.00
2,41.678063,2.243490,33.00,45.75,41.75,41.33,42.75,0.535979,0.405469,0.0,...,30.00,28.4575,31.25,0.383292,0.389164,0.0,1.79,0.43,0.00,0.50
3,43.454958,1.386098,37.00,48.00,43.25,42.50,45.00,0.378083,0.315566,0.0,...,23.50,22.2500,24.00,0.679646,0.622534,0.0,5.26,0.50,0.43,0.87
4,43.969125,1.618364,36.25,48.00,44.50,43.31,44.67,0.413125,0.263111,0.0,...,21.67,20.5000,23.75,0.555312,0.487826,0.0,2.96,0.49,0.00,0.83


(88, 42)


In [5]:
# 1ciii std and 90% bootstrap CI 
feature_std = feature_df.std()
display(feature_std)

import numpy as np
np.random.seed(42) 
n_bootstrap = 1000
bootstrap_stds = []

for i in range(n_bootstrap):
    sample = feature_df.sample(n=len(feature_df), replace=True)
    bootstrap_stds.append(sample.std().values)

bootstrap_stds = np.array(bootstrap_stds)
lower = np.percentile(bootstrap_stds, 5, axis=0)
upper = np.percentile(bootstrap_stds, 95, axis=0)  

std_results = pd.DataFrame({
    "Feature": feature_df.columns,
    "Standard Deviation": feature_df.std().values,
    "90% CI Lower": lower,
    "90% CI Upper": upper})

display(std_results.round(3))

sensor_1_mean      5.335718
sensor_1_std       1.772153
sensor_1_min       9.569975
sensor_1_max       4.394362
sensor_1_median    5.440054
sensor_1_q1        6.153590
sensor_1_q3        5.138925
sensor_2_mean      1.574164
sensor_2_std       0.884105
sensor_2_min       0.000000
sensor_2_max       5.062729
sensor_2_median    1.412244
sensor_2_q1        0.946386
sensor_2_q3        2.125266
sensor_3_mean      4.008380
sensor_3_std       0.946710
sensor_3_min       2.956462
sensor_3_max       4.875137
sensor_3_median    4.036396
sensor_3_q1        4.220658
sensor_3_q3        4.171628
sensor_4_mean      1.166114
sensor_4_std       0.458242
sensor_4_min       0.000000
sensor_4_max       2.183625
sensor_4_median    1.145586
sensor_4_q1        0.843620
sensor_4_q3        1.552504
sensor_5_mean      5.675593
sensor_5_std       1.024898
sensor_5_min       6.124001
sensor_5_max       5.741238
sensor_5_median    5.813782
sensor_5_q1        6.096465
sensor_5_q3        5.531720
sensor_6_mean      1

,Feature,Standard Deviation,90% CI Lower,90% CI Upper
0,sensor_1_mean,5.336,4.696,5.851
1,sensor_1_std,1.772,1.570,1.948
2,sensor_1_min,9.570,8.238,10.737
3,sensor_1_max,4.394,3.362,5.352
4,sensor_1_median,5.440,4.793,5.990
5,sensor_1_q1,6.154,5.565,6.624
6,sensor_1_q3,5.139,4.290,5.845
7,sensor_2_mean,1.574,1.407,1.704
8,sensor_2_std,0.884,0.807,0.943
9,sensor_2_min,0.000,0.000,0.000


Three features
From the analysis completed thus far, I've decided to choose the mean, standard deviation, and maximum as the most important time-domain features. The mean represents the average sensor measurement and also provides overall level of activity. The SD measures the variability of sensor readings, which could wid in distinguishing activities with stable measurements from findings with greater fluctuations. Lastly, the maximum identifies the highest value and therefore can help identify activities with larger sensor measurements based on this baseline.

ISLR 3.7.4
a - The cubic regression would have a training RSS less than or equal to the linear regression, as this model would include all of the possibilities of the linear model. Since it is more flexible, the training observations can be fitted even when the true relationship is linear, meaning the RSS can't be greater.

b - The linear regression would have a lower test RSS. This is the conclusion as a linear model would represent the underlying relationship correctly. The cubic model is overly complicated for this instance, which would cause overfitting.

c - The cubic regression would have a training RSS less than or equal to the linear regression. Knowing the cubic model is more flexible. the cubic regression can't have a higher training RSS than the linear regression.

d - There isn't enough information to determine the model with a lower test RSS.

In [ ]:
# question 4

train_features = feature_df.iloc[:len(train_files)].copy()
train_labels = [ 1 if file.parent.name in ["bending1", "bending2"] else 0 for file in train_files]
train_features['class'] = train_labels
print("Training data shape:", train_features.shape)
print("Class distribution:")
print(train_features['class'].value_counts())


#selected features (mean, std, max)
selected_features = []
for i in [1, 2, 6]:
    selected_features.extend([
        f"sensor_{i}_mean",
        f"sensor_{i}_std",
        f"sensor_{i}_max" ])
plot_data = train_features[selected_features + ["class"]]
display(plot_data.head())

import seaborn as sns
import matplotlib.pyplot as plt

sns.pairplot(plot_data, vars=selected_features, hue="class", palette={0: "blue", 1: "orange"}, diag_kind="hist")
plt.show()


In [ ]:
#4aii

split_rows = []
for file in train_files:
    df = load_arem_file(file)
    sensor_data = df.iloc[:, 1:]
    row = {}

    for i, col in enumerate(sensor_data.columns, start=1):
        values = sensor_data[col]
        segments = np.array_split(values.to_numpy(), 2)

        for j, segment in enumerate(segments, start=1):
            row[f"sensor_{i}_part{j}_mean"] = np.mean(segment)
            row[f"sensor_{i}_part{j}_std"] = np.std(segment, ddof=1)
            row[f"sensor_{i}_part{j}_max"] = np.max(segment)
    row["class"] = (
        1 if file.parent.name in ["bending1", "bending2"] else 0)
    split_rows.append(row)
split_features = pd.DataFrame(split_rows)
print("Shape:", split_features.shape)
display(split_features.head())

# scatterplot portion
selected_split_features = []
for i in [1, 2, 6]:
    for j in [1, 2]:
        selected_split_features.extend([
            f"sensor_{i}_part{j}_mean",
            f"sensor_{i}_part{j}_std",
            f"sensor_{i}_part{j}_max"])
sns.pairplot(split_features, vars= selected_split_features, hue="class", palette={0: "blue", 1: "orange"}, diag_kind="hist")

#commentary
The second pairplot was formed from extracting the mean, standard deviation, and maximum from each segment for a new matrix. The comparison is illustrated in additional plots but there are no considerable improvement between bending and nonbending activities. Both matrixes display some distinct clustering and overlap between the two classes. Although splitting the time series provides more detailed information, the additional features may still be useful for classification.

In [ ]:
#4a iii
def extract_segment_features(files, l):
    rows = []
    labels = []
    for file in files:
        df = load_arem_file(file)
        sensor_data = df.iloc[:, 1:]
        row = {}
        for i, col in enumerate(sensor_data.columns, start=1):
            values = sensor_data[col]
            segments = np.array_split(values.to_numpy(), l)
            for j, segment in enumerate(segments, start=1):
                row[f"sensor_{i}_part{j}_min"] = np.min(segment)
                row[f"sensor_{i}_part{j}_max"] = np.max(segment)
                row[f"sensor_{i}_part{j}_mean"] = np.mean(segment)
                row[f"sensor_{i}_part{j}_median"] = np.median(segment)
                row[f"sensor_{i}_part{j}_std"] = np.std(segment, ddof=1)
                row[f"sensor_{i}_part{j}_q1"] = np.percentile(segment, 25)
                row[f"sensor_{i}_part{j}_q3"] = np.percentile(segment, 75)
        rows.append(row)
        labels.append(1 if file.parent.name in ["bending1", "bending2"] else 0)
    X = pd.DataFrame(rows)
    y = pd.Series(labels)
    return X, y


# training and testing function 
X_train1, y_train1 = extract_segment_features(train_files, 1)
X_train2, y_train2 = extract_segment_features(train_files, 2)
print("l=1", X_train1.shape)
print("l=2", X_train2.shape)
print("Labels:", np.bincount(y_train1))

# setup
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42)
print(cv)


results = []
p_values = [1,2,3,5,10,15,20]

for l in range(1,21):
    X,y = extract_segment_features(train_files, l)
    for p in p_values:
        if p > X.shape[1]:
            continue
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("rfe", RFE(estimator=LogisticRegression(max_iter=1000, C=1e6),
                n_features_to_select=p,
                step=0.2 )),
            ("logistic", LogisticRegression(
                max_iter=1000,
                C=1e6 ))  ])

        scores = cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring="accuracy" )

        results.append({
            "l": l,
            "p": p,
            "CV Accuracy": scores.mean(),
            "CV Std": scores.std() })

cv_results = pd.DataFrame(results)
display(
    cv_results.sort_values(
        "CV Accuracy",
        ascending=False ).head(10))

best_result = cv_results.sort_values(
    ["CV Accuracy", "p", "l"],
    ascending=[False, True, True]
).iloc[0]

best_l = int(best_result["l"])
best_p = int(best_result["p"])
print(f"Best l: {best_l}, Best p: {best_p}")
print("Best CV Accuracy:", round(best_result["CV Accuracy"], 4))

# right versus wrong way
The right method to perform cross-validation is to apply feature selection separately within each training fold and evaluate the model on the corresponding validation fold. By doing this, it prevents information from the validation data from indluencing feature selection. The wrong method would be to select features from the entire dataset before cross-validation, because this could cause data leaks and conclude with overly optimistic accuracy estimates. The utilization of a pipeline ensures standardization, feature selection, and model fitting are performed within each training fold.

In [ ]:
#4a iv
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
import matplotlib.pyplot as plt

X_train, y_train = extract_segment_features(train_files, best_l)
best_model = Pipeline([
    ("scaler", StandardScaler()),
    ("rfe", RFE(estimator=LogisticRegression(max_iter=1000, C=1e6),
        n_features_to_select=best_p,
        step=0.2 )),
    ("logistic", LogisticRegression(
        max_iter=1000,
        C=1e6 ))  ])

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_train)
y_prob = best_model.predict_proba(X_train)[:,1]

print("Confusion Matrix:")
print(confusion_matrix(y_train, y_pred))
print("Training AUC:", roc_auc_score(y_train, y_prob))
fpr, tpr, thresholds = roc_curve(y_train, y_prob)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_train, y_prob):.4f}")
plt.plot([0,1], [0,1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Training ROC Curve")
plt.legend()
plt.show()


#4a iv coefficients
selected_features = X_train.columns[best_model.named_steps["rfe"].support_]
coefficients = best_model.named_steps["logistic"].coef_[0]
coef_df = pd.DataFrame({
    "Feature": selected_features,
    "Coefficient": coefficients})
display(coef_df)

# p-values
import statsmodels.api as sm
X_scaled = best_model.named_steps["scaler"].transform(X_train)
X_selected = X_scaled[:, best_model.named_steps["rfe"].support_]
X_sm = sm.add_constant(X_selected)
logit_model = sm.Logit(y_train, X_sm)
logit_result = logit_model.fit(disp=False)
print(logit_result.summary())

#4a iv interpretation
The logistic regression classifier correctly classified all 69 training instances. Additionally, the ROC produced an AUC of 1.0, illustrating perfect discrimination between bending and nonbending activities on the training set. But the lower cross-validation accuracy seems to suggest that performance on unseen observations might not be perfect. 

Three features interpretation
The three selected features were selected by the logistic regression from the first portion of the time series. sensor_3_part1_min has a negative coefficient, meaning that higher values are associated with a lower predicted probability of bending. Whereas sensor_5_part1_min and its max have positive coefficients, which indicates that higher values are associated with a greater predicted probability of bending -- holding the other selected features constant. These coefficients appear to be corresponding to standardized features. 
p-values interpretation

The logistic regression model acheived perfect classification for the training set, as seen by the confusion matrix of [[60,0], [0,9]] and an AUC of 1.0. But the statsmodels regression showed a complete separation and failed to converge. In conclusion, the max likelihood coefficient estimates aren't identifiable and the p-values are also unreliable. Based off this, the statistical significance cannot be determined using conventional p-values.

In [ ]:
# 4a v
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
X_test, y_test = extract_segment_features(test_files, best_l)
y_test_pred = best_model.predict(X_test)
y_test_prob = best_model.predict_proba(X_test)[:,1]
test_accuracy = accuracy_score(y_test, y_test_pred)
test_error = 1 - test_accuracy
print("test accuracy:", round(test_accuracy, 4))
print("test error:", round(test_error, 4))  
print("test confusion matrix: ")
print(confusion_matrix(y_test, y_test_pred))
print("test AUC:", round(roc_auc_score(y_test, y_test_prob), 4))

interpretation
The logistic regression model acheived a test accuracy of 94.74% with a test error of 5.26%. The 15 nonbending instances were identified correctly, but 3 of the 4 bending instances were correctly classified. One bending instance was incorrectly identified as nonbending. The slightly lower test accuracy suggests that the model generalizes well to the held out observations, although the test set is small.

In [ ]:
#4b 
from sklearn.model_selection import GridSearchCV
l1_results = []
C_values = [0.001,0.01,0.1,1,10,100]
for l in range(1,21):
    X,y = extract_segment_features(train_files, l)
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic", LogisticRegression(
            penalty="l1",
            solver="liblinear",
            max_iter=1000 ))  ])
    grid = GridSearchCV(
        model,
        {"logistic__C": C_values},
        cv=cv,
        scoring="accuracy" )
    grid.fit(X,y)
    l1_results.append({
        "l": l,
        "Best C": grid.best_params_["logistic__C"],
        "CV Accuracy": grid.best_score_ })
l1_cv_results = pd.DataFrame(l1_results)
display(l1_cv_results.sort_values("CV Accuracy", ascending=False).head(10))


# testing 
X_train_l1, y_train_l1 = extract_segment_features(train_files, 19)
X_test_l1, y_test_l1 = extract_segment_features(test_files, 19)

best_l1_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=1,
        max_iter=5000 ))  ])

best_l1_model.fit(X_train_l1, y_train_l1)
y_pred_l1 = best_l1_model.predict(X_test_l1)
y_prob_l1 = best_l1_model.predict_proba(X_test_l1)[:,1]

print("Test Accuracy:", round(accuracy_score(y_test_l1, y_pred_l1), 4))
print("Test Error:", round(1 - accuracy_score(y_test_l1, y_pred_l1), 4))
print("Confusion Matrix:")
print(confusion_matrix(y_test_l1, y_pred_l1))
print("Test AUC:", round(roc_auc_score(y_test_l1, y_prob_l1), 4))

In [ ]:
#4c i testing portion 
best_l_multi = 1
best_C_multi = 1

X_train_multi, y_train_multi = extract_multiclass_features(train_files, best_l_multi)
X_test_multi, y_test_multi = extract_multiclass_features(test_files, best_l_multi)

multi_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        penalty="l1",
        solver="saga",
        C=best_C_multi,
        max_iter=1000,
        tol=0.01,
        random_state=42 ))  ])

multi_model.fit(X_train_multi, y_train_multi)
y_pred_multi = multi_model.predict(X_test_multi)
y_prob_multi = multi_model.predict_proba(X_test_multi)

print("Test Accuracy:", round(accuracy_score(y_test_multi, y_pred_multi), 4))
print("Test Error:", round(1 - accuracy_score(y_test_multi, y_pred_multi), 4))
print("Confusion Matrix:")
print(confusion_matrix(y_test_multi, y_pred_multi,
    labels=multi_model.named_steps["logistic"].classes_))

interpretation
The L1-penalized multinomial logistic regression model acheived a cross-validation accuracy of 92.81% and a test accuracy of 89.47% corresponding to a test error of 10.53%. The confusion matrix shows that 17 out of 19 instances were identified correctly. The two misclassifications happened between the sitting and standing classes, which suggests that these two have similar sensor characteristics and are more difficult for the mode

In [ ]:
#4c i multiclass ROC
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

classes = multi_model.named_steps["logistic"].classes_
y_test_bin = label_binarize(y_test_multi, classes=classes)
plt.figure(figsize=(8,6))

for i, activity in enumerate(classes):
    fpr, tpr, thresholds = roc_curve(
        y_test_bin[:,i],
        y_prob_multi[:,i] )
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{activity} (AUC = {roc_auc:.4f})")

plt.plot([0,1], [0,1], "--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multiclass Logistic Regression ROC Curves")
plt.legend()
plt.show()

interpretation
The L1-penalized multinomial logistic regression achieved a test accuracy of 89.47% and a test error 10.53%. The one-vs-rest ROC curve showed AUC values of 1.0 for five activities, while sitting and standing achieved AUC values of 0.9792 and 0.9583, respectively. The results indicate strong overall classification performance, with sitting and standing being the most difficult activities to distinguish.

In [ ]:
#4c ii Gaussian Naive Bayes
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix

gaussian_model = GaussianNB()
gaussian_model.fit(X_train_multi, y_train_multi)
gaussian_pred = gaussian_model.predict(X_test_multi)

print("Gaussian NB Test Accuracy:", round(accuracy_score(y_test_multi, gaussian_pred), 4))
print("Gaussian NB Test Error:", round(1 - accuracy_score(y_test_multi, gaussian_pred), 4))
print("Gaussian NB Confusion Matrix:")
print(confusion_matrix(y_test_multi, gaussian_pred, labels=classes))


# 4c ii continuied 
#4c ii Multinomial Naive Bayes
from sklearn.preprocessing import MinMaxScaler
multinomial_model = Pipeline([
    ("scaler", MinMaxScaler()),
    ("naive_bayes", MultinomialNB())  ])
multinomial_model.fit(X_train_multi, y_train_multi)
multinomial_pred = multinomial_model.predict(X_test_multi)

print("Multinomial NB Test Accuracy:", round(accuracy_score(y_test_multi, multinomial_pred), 4))
print("Multinomial NB Test Error:", round(1 - accuracy_score(y_test_multi, multinomial_pred), 4))
print("Multinomial NB Confusion Matrix:")
print(confusion_matrix(y_test_multi, multinomial_pred, labels=classes))


# l=1 CV 
from sklearn.model_selection import cross_val_score
nb_results = []

for l in range(1,21):
    X,y = extract_multiclass_features(train_files, l)

    gaussian_model = GaussianNB()
    gaussian_scores = cross_val_score(
        gaussian_model,
        X,
        y,
        cv=cv_multi,
        scoring="accuracy" )

    multinomial_model = Pipeline([
        ("scaler", MinMaxScaler()),
        ("naive_bayes", MultinomialNB()) ])

    multinomial_scores = cross_val_score(
        multinomial_model,
        X,
        y,
        cv=cv_multi,
        scoring="accuracy" )

    nb_results.append({
        "l": l,
        "Gaussian CV Accuracy": gaussian_scores.mean(),
        "Multinomial CV Accuracy": multinomial_scores.mean() })

nb_cv_results = pd.DataFrame(nb_results)
display(nb_cv_results.sort_values(
    "Gaussian CV Accuracy",
    ascending=False ).head(10))

#best gaussian
best_gaussian = nb_cv_results.sort_values(
    "Gaussian CV Accuracy",
    ascending=False ).iloc[0]

best_multinomial = nb_cv_results.sort_values(
    "Multinomial CV Accuracy",
    ascending=False ).iloc[0]

print("Best Gaussian l:", int(best_gaussian["l"]))
print("Gaussian CV Accuracy:", round(best_gaussian["Gaussian CV Accuracy"], 4))
print("Best Multinomial l:", int(best_multinomial["l"]))
print("Multinomial CV Accuracy:", round(best_multinomial["Multinomial CV Accuracy"], 4))

In [ ]:
#4c ii final steps 

# Gaussian NB
X_train_g, y_train_g = extract_multiclass_features(train_files, 1)
X_test_g, y_test_g = extract_multiclass_features(test_files, 1)

gaussian_model = GaussianNB()
gaussian_model.fit(X_train_g, y_train_g)
gaussian_pred = gaussian_model.predict(X_test_g)

print("Gaussian NB Test Accuracy:", round(accuracy_score(y_test_g, gaussian_pred), 4))
print("Gaussian NB Test Error:", round(1 - accuracy_score(y_test_g, gaussian_pred), 4))
print("Gaussian NB Confusion Matrix:")
print(confusion_matrix(y_test_g, gaussian_pred, labels=classes))


# Multinomial NB
X_train_m, y_train_m = extract_multiclass_features(train_files, 5)
X_test_m, y_test_m = extract_multiclass_features(test_files, 5)

multinomial_model = Pipeline([
    ("scaler", MinMaxScaler()),
    ("naive_bayes", MultinomialNB())  ])

multinomial_model.fit(X_train_m, y_train_m)
multinomial_pred = multinomial_model.predict(X_test_m)

print("Multinomial NB Test Accuracy:", round(accuracy_score(y_test_m, multinomial_pred), 4))
print("Multinomial NB Test Error:", round(1 - accuracy_score(y_test_m, multinomial_pred), 4))
print("Multinomial NB Confusion Matrix:")
print(confusion_matrix(y_test_m, multinomial_pred, labels=classes))

#4c iii interpretation
L1-penalized multinomial logistic regression achieved the highest cross-validation accuracy of 92.81%, compared to 84.15% for Guassian Naive Bayes and 78.27% for Multinomial Naive Bayes. On the test set, both the logistic regression and Gaussian Naive Bayes acheived the same accuracy of 89.47% and a test error of 10.53%; while Multinomial Naive Bayes achieved a lower accuracy of 78.95% and a test error of 21.05%. 

In conclusion, L1-penalized multinomial logistic regression performed best during the CV while tied with Gaussian Naive Bayes on test accuracy. Gaussian Naive Bayes was also effective, with the Multinomial Naive Bayes performing the worst. A possible explanation would be the Gaussian Naive Bayes is more suitable for continuous sensor features, while the Multinomial Naive Bayes is made for count-based instances/datasets.